Run with shape: (1027368, 18)  
gva1 L1.gva1 total_assets employees tfp_wav1 | gmm(gva1, 2:4) gmm(total_assets, 2:3) iv(tfp_wav1) | timedumm  
- With 'collapse': takes just over 1 minute to run

In [1]:
# Install dependencies if needed:
# !pip install "numpy<2.0.0" "pandas<2.2.0" pydynpd

import pandas as pd
import numpy as np
from pydynpd import regression
from typing import Any

print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

if np.__version__ >= "2.0.0":
    raise ImportError("NumPy version must be less than 2.0.0")
if pd.__version__ >= "2.2.0":
    raise ImportError("Pandas version must be less than 2.2.0")
print("Libraries loaded successfully.")

NumPy version: 1.26.4
Pandas version: 2.1.4
Libraries loaded successfully.


In [7]:
import pandas as pd
from utils.f_0_dirs import get_data_dirs

dirs = get_data_dirs(segment="model")

table_panel_name = "working_yearly_n"
parquet_path = dirs.tmp_dir / f"{table_panel_name}.parquet"
print(f"Parquet file path: {parquet_path}")
df_panel = pd.read_parquet(parquet_path)
if df_panel is None:
    print(f"Failed to load DataFrame from {parquet_path}")
    df_panel = pd.read_csv(dirs.tmp_dir / f"{table_panel_name}.csv")
if df_panel is None:
    raise FileNotFoundError(f"Could not load DataFrame from {parquet_path} or CSV.")
# log_vars = ["gva1", "total_assets", "employees"]
df_panel_dyn = (
    df_panel
    .dropna(subset=["y", "k", "l"])
    .drop_duplicates(subset=["registered_number", "year"], keep="last")
    # .assign(**{
    #     f'ln_{p}': df_panel[p].apply(lambda x: np.log(x) if x > 0 else None) for p in log_vars
    # })
)
print(f"DataFrame loaded successfully with shape: {df_panel_dyn.shape}")

Parquet file path: C:\Users\lazyst\Files\ucl\Dissertation\model\tmp\working_yearly_n.parquet
DataFrame loaded successfully with shape: (1045164, 35)


In [5]:
def is_number(s: Any) -> bool:
    try:
        float(s)
        return True
    except (ValueError, TypeError):
        return False

def unpack_x_var(var_tuple: tuple[str, ...]) -> str:
    var, *lags = var_tuple
    if len(lags) == 0:
        return var
    elif len(lags) == 1:
        if lags[0] == 0:
            return var
        else:
            return f"L{lags[0]}.{var}"
    elif len(lags) == 2:
        if lags[0] == lags[1]:
            return f"L{lags[0]}.{var}"
        else:
            return f"L({lags[0]}:{lags[1]}).{var}"
    else:
        raise ValueError(f"Too many lags specified for variable {var}: {lags}")

def unpack_z_var(var_tuple: tuple[str, ...]) -> str:
    var, *lags = var_tuple
    if len(lags) == 0:
        return f"iv({var})"
    elif len(lags) == 1:
        if lags[0] == 0:
            return f"iv({var})"
        elif lags[0] == 1:
            return f"pred({var})"
        else:
            return f"gmm(L{lags[0]}.{var})"
    elif len(lags) == 2:
        if lags[0] == lags[1]:
            if lags[0] == 0:
                return f"iv({var})"
            elif lags[0] == 1:
                return f"pred({var})"
            else:
                return f"gmm(L{lags[0]}.{var})"
        elif is_number(lags[0]) and is_number(lags[1]) and int(lags[0]) > int(lags[1]):
            raise ValueError(f"Invalid lag range for variable {var}: {lags}")
        else:
            return f"gmm({var}, {lags[0]}:{lags[1]})"
    else:
        raise ValueError(f"Too many lags specified for variable {var}: {lags}")

# PYDYNPD package

### Part 1: dependent-independent
- Basic: `n L1.n L2.n w k`. Don't distinguish between endogenous and predetermined variables here.
- `n`: my `ln(y)`, dependent variable

### Part 2: instrument creation
- `gmm(2:5)`. Use 2: if the variable is endogenous, 1: if predetermined.
- `.`: dot is no restriction on maximum lags.
- `?`: automatic mode: find maximum lags.
- `endo(list of variables)` is equivalent to `gmm(list of variables, 2:.)`
- `pred(list of variables)` is equivalent to `gmm(list of variables, 1:.)`
- `iv(k)`: strictly exogenous variable.

### Test results:
- `AR(1) test reject null`: first-order autocorrelation. Cannot use 1st lag of DV as instrument.
- `AR(2) test accept null`: no 2nd-order autocorrelation. Fine to use 2nd lag of DV as instrument, don't use as a regressor.
- `Hansen J reject`: instruments are not exogenous (overidentified)
  
### Ex: command strings
- `ln_gva1 L(1:1).ln_gva1 ln_total_assets ln_employees tfp_wav1 | gmm(ln_gva1, 2:5) gmm(ln_total_assets, 2:4) iv(tfp_wav1) | timedumm collapse`
--------------------------------------------------

In [14]:
models = [
    # Different AR lags on y
    {
        "Y": ("y", 0),
        "X": [
            ("y", 1),
            ("k", 0),
            ("l", 0),
            ("wd1_y", 0, 1),
            ("wd1_k", 0, 1),
            ("wd1_l", 0, 1)
        ],
        "Z": [
            ("y", 2, 5),
            ("k", 3, 5),
            ("l", 3, 5),
            ("wd1_y", 3, 6),
            ("wd1_k", 3, 6),
            ("wd1_l", 3, 6)
        ],
        "options": ["timedumm", "collapse"]
    },
    # Different AR lags on y
    {
        "Y": ("y", 0),
        "X": [
            ("y", 1, 2),
            ("k", 0),
            ("l", 0),
            ("wd1_y", 0, 1),
            ("wd1_k", 0, 1),
            ("wd1_l", 0, 1)
        ],
        "Z": [
            ("y", 2, 5),
            ("k", 3, 5),
            ("l", 3, 5),
            ("wd1_y", 3, 6),
            ("wd1_k", 3, 6),
            ("wd1_l", 3, 6)
        ],
        "options": ["timedumm", "collapse"]
    },
    {
        "Y": ("y", 0),
        "X": [
            ("y", 1),
            ("k", 0),
            ("l", 0),
            ("wd1_y", 0, 1),
            ("wd1_k", 0, 1),
            ("wd1_l", 0, 1)
        ],
        "Z": [
            ("y", 2, 4),
            ("k", 3, 4),
            ("l", 3, 4),
            ("wd1_y", 3, 4),
            ("wd1_k", 3, 4),
            ("wd1_l", 3, 4)
        ],
        "options": ["timedumm", "collapse"]
    }
]
    
for mod in models:

    y_var = mod["Y"][0]
    x_vars = [unpack_x_var(x) for x in mod["X"]]
    z_vars = [unpack_z_var(z) for z in mod["Z"]]

    combined = [mod["Y"]] + mod["X"] + mod["Z"]
    all_vars = [var[0] for var in combined]
    df_filtered = df_panel_dyn.copy().dropna(subset=all_vars)

    command_str = f"{y_var} {' '.join(x_vars)} | {' '.join(z_vars)} | {' '.join(mod['options'])}"
    print("Generated Command String:")
    print(command_str)
    print("-" * 50)

    sys_gmm_model = regression.abond(command_str, df_filtered, ['registered_number', 'year'])

Generated Command String:
y L1.y k l L(0:1).wd1_y L(0:1).wd1_k L(0:1).wd1_l | gmm(y, 2:5) gmm(k, 3:5) gmm(l, 3:5) gmm(wd1_y, 3:6) gmm(wd1_k, 3:6) gmm(wd1_l, 3:6) | timedumm collapse
--------------------------------------------------
 Dynamic panel-data estimation, two-step system GMM
 Group variable: registered_number                       Number of obs = 686164  
 Time variable: year                                     Min obs per group: 0    
 Number of instruments = 45                              Max obs per group: 16   
 Number of groups = 139638                               Avg obs per group: 4.91 
+-----------+------------+---------------------+------------+-----------+-----+
|     y     |   coef.    | Corrected Std. Err. |     z      |   P>|z|   |     |
+-----------+------------+---------------------+------------+-----------+-----+
|    L1.y   | -0.0000818 |      0.0173328      | -0.0047184 | 0.9962352 |     |
|     k     | 0.4311287  |      0.0702277      | 6.1390131  | 0.000

In [2]:
%%script old config

# 1. Define your variables programmatically
y_var = "ln_gva1"
lag_dep_var = f"L(1:{ar_lag}).{y_var}"
standard_factors = ["ln_total_assets", "ln_employees"]
instr_standard = standard_factors[0]
z_var = "tfp_wav1"
z_var_lag = "" if z_lag == 0 else f" L(1:{z_lag}).tfp_wav1"

# 2. Join the list of standard factors with spaces, and adds the other variables
structural_eq = f"{y_var} {lag_dep_var} {' '.join(standard_factors)} {z_var}"

# 3. GMM instruments for endogenous/predetermined vars
# Standard IVs for strictly exogenous vars
gmm_inst = f"gmm({y_var}, 2:5) gmm({instr_standard}, 2:4)"
iv_inst = f"iv({z_var}{z_var_lag})"

# 4. Build the Options
options_arr = ["timedumm", "collapse"]
options = " ".join(options_arr)

# 5. Concatenate everything using the pydynpd pipe '|' syntax
command_str = f"{structural_eq} | {gmm_inst} {iv_inst} | {options}"

# Let's print it to verify it looks exactly right before running
print("Generated Command String:")
print(command_str)
print("-" * 50)

# Execute estimation
# sys_gmm_model = regression.abond(command_str, df_filtered, ['registered_number', 'year'])

Couldn't find program: 'old'
